In [1]:
import itertools
import numpy as np
import pandas as pd
from dtw import dtw
import matplotlib.pyplot as plt
import math
import re
import base_functions as bf

from ipywidgets import interact, widgets, fixed

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
xl_file = pd.ExcelFile('I_full data.xlsx')

dfs = {sheet_name: xl_file.parse(sheet_name) 
          for sheet_name in xl_file.sheet_names}

In [3]:
dfs['B'].columns

Index(['thickness, mm', 'Piece', 'sample №', 'thick, mm', 'weight, g',
       'Vol, cm3', 'depth, mm', 'Dgeo', 'Igeo', 'MADgeo', 'NRMmax (Am2/kg)',
       'ARMmax (Am2/kg)'],
      dtype='str')

In [4]:
section_names = [
    "A",
    "B",
    "C",
    "D",
    #"E",
    #"F",
]

params = [
    "Dgeo",
    "Igeo",
    #"NRMmax (Am2/kg)",
    "ARMmax (Am2/kg)",
]

skips = {
    "A": 1,
    "B": 1,
    "C": 1,
    "D": 1,
    #"E": 1,
    #"F": 1,
}

In [5]:
picture_configuration = 4

param_anc = 'Igeo'

max_len = max(
    len(dfs[name].iloc[skips.get(name, 0):])
    for name in section_names
)

interact(
    bf.dynamic_draw_sections,
    dfs=fixed(dfs),
    section_names=fixed(section_names),
    param_name=fixed(param_anc),
    skips=fixed(skips),
    x_pos=widgets.IntSlider(
        min=0,
        max=max_len - 1,
        step=1,
        value=0,
        description="Index"
    ),
    ncols=fixed(picture_configuration)
)

param_anc = 'Dgeo'
max_len = max(
    len(dfs[name].iloc[skips.get(name, 0):])
    for name in section_names
)

interact(
    bf.dynamic_draw_sections,
    dfs=fixed(dfs),
    section_names=fixed(section_names),
    param_name=fixed(param_anc),
    skips=fixed(skips),
    x_pos=widgets.IntSlider(
        min=0,
        max=max_len - 1,
        step=1,
        value=0,
        description="Index"
    ),
    ncols=fixed(picture_configuration)
)

param_anc = 'ARMmax (Am2/kg)'
max_len = max(
    len(dfs[name].iloc[skips.get(name, 0):])
    for name in section_names
)

interact(
    bf.dynamic_draw_sections,
    dfs=fixed(dfs),
    section_names=fixed(section_names),
    param_name=fixed(param_anc),
    skips=fixed(skips),
    x_pos=widgets.IntSlider(
        min=0,
        max=max_len - 1,
        step=1,
        value=0,
        description="Index"
    ),
    ncols=fixed(picture_configuration)
)

interactive(children=(IntSlider(value=0, description='Index', max=36), Output()), _dom_classes=('widget-intera…

interactive(children=(IntSlider(value=0, description='Index', max=36), Output()), _dom_classes=('widget-intera…

interactive(children=(IntSlider(value=0, description='Index', max=36), Output()), _dom_classes=('widget-intera…

<function base_functions.dynamic_draw_sections(dfs, section_names, param_name, x_pos, skips=None, ncols=3)>

In [6]:
anchors_CD = [
    (0, 0),
    (4, 4),
    (14, 16),
    (29, 31),
]

In [7]:
anchors_BD = [
    (0, 0),
    (3, 4),
    (13, 16),
    (29, 31),
]

In [8]:
anchors_BC = [
    (0, 0),
    (3, 4),
    (13, 14),
    (29, 29),
]

In [9]:
anchors_AD = [
    (0,0),
    (4,4),
    (12,16),
    (30,31),
]

In [10]:
anchors_AC = [
    (0,0),
    (4,4),
    (12,14),
    (30,29),
]

In [11]:
anchors_AB =[
    (0, 0),
    (4,3),
    (12,13),
    (30,29),    
]

In [12]:
anchors_by_pair = {
    ("A","B"): anchors_AB,
    ("A", "C"): anchors_AC,
    ("A", "D"):  anchors_AD,
    ("B", "C"): anchors_BC,
    ("B", "D"): anchors_BD,
    ("C", "D"): anchors_CD,    
}

In [13]:
sections = bf.make_sections(
    dfs=dfs,
    section_names=section_names,
    params=params,
    skips=skips,
)

In [14]:
for name, arr in sections.items():
    print(name, arr.shape, "NaNs:", np.isnan(arr).sum())

A (37, 3) NaNs: 22
B (37, 3) NaNs: 9
C (36, 3) NaNs: 9
D (36, 3) NaNs: 10


In [15]:
weights = {
    "Dgeo": 1.0,
    "Igeo": 1.0,
    #"NRMmax (Am2/kg)": 0.25,
    "ARMmax (Am2/kg)": 1.0,
}

In [16]:
scales = bf.compute_global_scales(
    sections,
    params=params,
    method="mad",
    scale_overrides={
        "Dgeo": 20.0,
        "Igeo": 10.0,
    },
)

pd.Series(scales, index=params)

Dgeo               2.000000e+01
Igeo               1.000000e+01
ARMmax (Am2/kg)    1.800257e-07
dtype: float64

In [17]:
distance_matrix_anchor, pair_alignments_anchor = bf.pairwise_dtw_matrix(
    sections=sections,
    params=params,
    weights=weights,
    scales=scales,
    anchors_by_pair=anchors_by_pair,
    circular_params=("Dgeo",),
    metric="l1",
    missing_penalty=0.0,
    step_pattern="symmetric2",
)


In [18]:
distance_matrix_anchor

,A,B,C,D
A,0.000000,1.236277,1.380251,1.334721
B,1.236277,0.000000,1.355878,1.383308
C,1.380251,1.355878,0.000000,1.135539
D,1.334721,1.383308,1.135539,0.000000
